In [3]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
CLEAN_CSV_PATH = "AirfoilDatset_cleaned_1.csv"   # <- change if your filename differs


In [4]:
# -----------------------------
# Cell 1: Load data
# -----------------------------
df = pd.read_csv(CLEAN_CSV_PATH)

print("Dataset shape:", df.shape)
print("Columns:", len(df.columns))
display(df.head())


C:\Users\peeye\AppData\Local\Temp\ipykernel_40692\2079294123.py:4: DtypeWarning: Columns (70) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CLEAN_CSV_PATH)


Dataset shape: (838207, 71)
Columns: 71


,airfoilName,upperSurfaceCoeff1,upperSurfaceCoeff2,upperSurfaceCoeff3,upperSurfaceCoeff4,upperSurfaceCoeff5,upperSurfaceCoeff6,upperSurfaceCoeff7,upperSurfaceCoeff8,upperSurfaceCoeff9,...,lowerSurfaceCoeff30,lowerSurfaceCoeff31,reynoldsNumber,alpha,coefficientLift,coefficientDrag,coefficientParasiteDrag,coefficientMoment,topXTR,botXTR
0,2032c,103000000.0,-74000000.0,-105000000.0,-51200000.0,29400000.0,88500000.0,96800000.0,50300000.0,-28100000.0,...,0.115,-0.00689,50000,-7.50,-0.2326,0.10640,0.10052,-0.0214,1.0,0.1007
1,2032c,103000000.0,-74000000.0,-105000000.0,-51200000.0,29400000.0,88500000.0,96800000.0,50300000.0,-28100000.0,...,0.115,-0.00689,50000,-7.25,-0.2270,0.10345,0.09765,-0.0208,1.0,0.1042
2,2032c,103000000.0,-74000000.0,-105000000.0,-51200000.0,29400000.0,88500000.0,96800000.0,50300000.0,-28100000.0,...,0.115,-0.00689,50000,-7.00,-0.2270,0.10172,0.09604,-0.0205,1.0,0.1074
3,2032c,103000000.0,-74000000.0,-105000000.0,-51200000.0,29400000.0,88500000.0,96800000.0,50300000.0,-28100000.0,...,0.115,-0.00689,50000,-6.75,-0.2298,0.10096,0.09544,-0.0212,1.0,0.1103
4,2032c,103000000.0,-74000000.0,-105000000.0,-51200000.0,29400000.0,88500000.0,96800000.0,50300000.0,-28100000.0,...,0.115,-0.00689,50000,-6.50,-0.2352,0.10161,0.09625,-0.0237,1.0,0.1117


In [5]:
# -----------------------------
# Cell 2: Define features/targets (FIXED)
# -----------------------------

# Drop airfoilName if it still exists (categorical)
drop_cols = []
if "airfoilName" in df.columns:
    drop_cols.append("airfoilName")

TARGET_CL = "coefficientLift"
TARGET_CD = "coefficientDrag"

# Safety checks
assert TARGET_CL in df.columns, f"Missing target column: {TARGET_CL}"
assert TARGET_CD in df.columns, f"Missing target column: {TARGET_CD}"

# Convert transition locations explicitly to numeric
for col in ["topXTR", "botXTR"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Drop rows where conversion failed (very few / none)
df = df.dropna(subset=["topXTR", "botXTR"])

# Define features and targets
X = df.drop(columns=[TARGET_CL, TARGET_CD] + drop_cols)
y_cl = df[TARGET_CL]
y_cd = df[TARGET_CD]

# Final check: ensure everything is numeric
obj_cols = X.select_dtypes(include=["object"]).columns.tolist()
if obj_cols:
    raise ValueError(f"Still found non-numeric columns in X: {obj_cols}")

print("X shape:", X.shape)
print("y_cl shape:", y_cl.shape, "y_cd shape:", y_cd.shape)


X shape: (838206, 68)
y_cl shape: (838206,) y_cd shape: (838206,)


In [6]:
# -----------------------------
# Cell 3: Train/Test split (80/20)
# -----------------------------
# Use a single split for both targets (same X_train/X_test)
X_train, X_test, y_cl_train, y_cl_test = train_test_split(
    X, y_cl, test_size=0.20, random_state=RANDOM_STATE
)

# IMPORTANT: create y_cd_train/y_cd_test using the same indices
y_cd_train = y_cd.loc[y_cl_train.index]
y_cd_test  = y_cd.loc[y_cl_test.index]

print("Train:", X_train.shape, "Test:", X_test.shape)


Train: (670564, 68) Test: (167642, 68)


In [7]:
# -----------------------------
# Cell 4: Helper functions (metrics) - FIX for older sklearn
# -----------------------------
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def regression_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)   # no squared parameter
    rmse = np.sqrt(mse)                        # RMSE manually
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

def print_metrics(title, y_true, y_pred):
    mae, rmse, r2 = regression_metrics(y_true, y_pred)
    print(f"\n{title}")
    print(f"MAE : {mae:.6f}")
    print(f"RMSE: {rmse:.6f}")
    print(f"R²  : {r2:.6f}")
    return mae, rmse, r2


In [12]:
# -----------------------------
# Cell 5: Base Random Forest (Cl) + CV + Test evaluation (Memory-safe)
# -----------------------------
rf_cl = RandomForestRegressor(
    n_estimators=200,
    random_state=RANDOM_STATE,
    n_jobs=-1          # RF parallel ON
)

cv = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

# IMPORTANT: CV parallel OFF to avoid nested parallelism
cv_r2_scores_cl = cross_val_score(
    rf_cl, X_train, y_cl_train,
    scoring="r2", cv=cv, n_jobs=1
)

cv_mae_scores_cl = cross_val_score(
    rf_cl, X_train, y_cl_train,
    scoring="neg_mean_absolute_error", cv=cv, n_jobs=1
)

cv_rmse_scores_cl = cross_val_score(
    rf_cl, X_train, y_cl_train,
    scoring="neg_root_mean_squared_error", cv=cv, n_jobs=1
)

print("Random Forest (Cl) - 3-Fold CV R² scores:", cv_r2_scores_cl)
print("Random Forest (Cl) - Mean CV R²:", cv_r2_scores_cl.mean())
print("Random Forest (Cl) - Mean CV MAE:", -cv_mae_scores_cl.mean())
print("Random Forest (Cl) - Mean CV RMSE:", -cv_rmse_scores_cl.mean())

# Fit and evaluate on test set
rf_cl.fit(X_train, y_cl_train)
y_cl_pred = rf_cl.predict(X_test)

base_cl_mae, base_cl_rmse, base_cl_r2 = print_metrics(
    "Base Random Forest Results (Cl) - Test Set", y_cl_test, y_cl_pred
)


Random Forest (Cl) - 3-Fold CV R² scores: [0.99799041 0.99792516 0.99796835]
Random Forest (Cl) - Mean CV R²: 0.9979613048515864
Random Forest (Cl) - Mean CV MAE: 0.021955952897743825
Random Forest (Cl) - Mean CV RMSE: 0.03343643263106289

Base Random Forest Results (Cl) - Test Set
MAE : 0.018546
RMSE: 0.028740
R²  : 0.998496


In [15]:
# -----------------------------
# Cell 6: Random Forest (Cd) - FAST evaluation (no CV to avoid MemoryError)
# -----------------------------
from sklearn.ensemble import RandomForestRegressor

rf_cd = RandomForestRegressor(
    n_estimators=120,      # keep small for speed
    random_state=RANDOM_STATE,
    n_jobs=2               # limit cores to reduce RAM usage
)

rf_cd.fit(X_train, y_cd_train)
y_cd_pred = rf_cd.predict(X_test)

base_cd_mae, base_cd_rmse, base_cd_r2 = print_metrics(
    "Base Random Forest Results (Cd) - Test Set", y_cd_test, y_cd_pred
)



Base Random Forest Results (Cd) - Test Set
MAE : 0.000297
RMSE: 0.000453
R²  : 0.999721


In [17]:
# -----------------------------
# Cell 7: Randomized tuning for Random Forest (Cl)
# -----------------------------
param_dist = {
    "rf__n_estimators": [200, 300, 500],
    "rf__max_depth": [None, 10, 20, 30],
    "rf__min_samples_split": [2, 5, 10],
    "rf__min_samples_leaf": [1, 2, 4],
    "rf__max_features": ["sqrt", "log2", None]
}

rf_cl_tune_pipeline = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("rf", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1))
])

random_search_cl = RandomizedSearchCV(
    estimator=rf_cl_tune_pipeline,
    param_distributions=param_dist,
    n_iter=15,               # keep reasonable
    scoring="r2",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

random_search_cl.fit(X_train, y_cl_train)

print("\nBest params (Cl):", random_search_cl.best_params_)
print("Best CV R² (Cl):", random_search_cl.best_score_)

tuned_cl_model = random_search_cl.best_estimator_
tuned_y_cl_pred = tuned_cl_model.predict(X_test)

tuned_cl_mae, tuned_cl_rmse, tuned_cl_r2 = print_metrics(
    "Tuned Random Forest Results (Cl) - Test Set", y_cl_test, tuned_y_cl_pred
)


Fitting 3 folds for each of 15 candidates, totalling 45 fits


: 

In [ ]:
# -----------------------------
# Cell 8: Randomized tuning for Random Forest (Cd)
# -----------------------------
rf_cd_tune_pipeline = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("rf", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1))
])

random_search_cd = RandomizedSearchCV(
    estimator=rf_cd_tune_pipeline,
    param_distributions=param_dist,
    n_iter=15,
    scoring="r2",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

random_search_cd.fit(X_train, y_cd_train)

print("\nBest params (Cd):", random_search_cd.best_params_)
print("Best CV R² (Cd):", random_search_cd.best_score_)

tuned_cd_model = random_search_cd.best_estimator_
tuned_y_cd_pred = tuned_cd_model.predict(X_test)

tuned_cd_mae, tuned_cd_rmse, tuned_cd_r2 = print_metrics(
    "Tuned Random Forest Results (Cd) - Test Set", y_cd_test, tuned_y_cd_pred
)



In [ ]:
# -----------------------------
# Cell 9: Final comparison tables (Base vs Tuned)
# -----------------------------
results_cl = pd.DataFrame([
    ["Base_RandomForest (Cl)",  base_cl_mae,  base_cl_rmse,  base_cl_r2],
    ["Tuned_RandomForest (Cl)", tuned_cl_mae, tuned_cl_rmse, tuned_cl_r2]
], columns=["Model", "MAE", "RMSE", "R²"])

results_cd = pd.DataFrame([
    ["Base_RandomForest (Cd)",  base_cd_mae,  base_cd_rmse,  base_cd_r2],
    ["Tuned_RandomForest (Cd)", tuned_cd_mae, tuned_cd_rmse, tuned_cd_r2]
], columns=["Model", "MAE", "RMSE", "R²"])

print("\n=== Random Forest Comparison (Cl) ===")
display(results_cl)

print("\n=== Random Forest Comparison (Cd) ===")
display(results_cd)

In [2]:
print("Cd variance:", np.var(y_cd_test))
print("Cd min/max:", y_cd_test.min(), y_cd_test.max())


NameError: name 'np' is not defined

In [8]:
# -----------------------------
# Cell X: Learning Curve for Random Forest (Cl)
# -----------------------------
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import learning_curve
from sklearn.ensemble import RandomForestRegressor

rf_cl_lc = RandomForestRegressor(
    n_estimators=100,      # reduced for speed
    random_state=RANDOM_STATE,
    n_jobs=2               # limit cores to avoid memory issues
)

train_sizes, train_scores, val_scores = learning_curve(
    estimator=rf_cl_lc,
    X=X_train,
    y=y_cl_train,
    train_sizes=np.linspace(0.2, 1.0, 5),  # fewer points = faster
    cv=3,
    scoring="r2",
    n_jobs=1                                # IMPORTANT: avoid nested parallelism
)

train_mean = np.mean(train_scores, axis=1)
val_mean = np.mean(val_scores, axis=1)

plt.figure()
plt.plot(train_sizes, train_mean, label="Training R²")
plt.plot(train_sizes, val_mean, label="Validation R²")
plt.xlabel("Training Set Size")
plt.ylabel("R² Score")
plt.title("Learning Curve – Random Forest (Cl)")
plt.legend()
plt.grid(True)
plt.show()


KeyboardInterrupt: 